## Packages

In [1]:
from DataProcessing.Plot import *

In [2]:
from Equipments import BNC575
from Equipments import Weeder
from Equipments import SR400
from Equipments import MCBOX
from Equipments import Andor
import numpy as np
import matplotlib.pyplot as plt
import pyvisa
import time

## Initialize

In [3]:
rm = pyvisa.ResourceManager()
print(rm.list_resources())

('TCPIP0::141.211.97.110::inst0::INSTR', 'ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL6::INSTR', 'ASRL7::INSTR', 'GPIB0::9::INSTR', 'GPIB0::23::INSTR')


In [4]:
# sr400 = SR400.SR400('GPIB0::23::INSTR')
sr400 = SR400.SR400('GPIB0::23::INSTR',timeout=1000)

In [5]:
bnc575 = BNC575.BNC575("GPIB0::9::INSTR")

In [ ]:
# set the pulse sequence
#                   channel width  delay  output
pulse_arrangement = [["A", 9000e-6], # MOT
                     ["B", 4e-6, 9001e-6], # 480 EXC
                     ["C", 4e-6, 9001e-6], # 780 EXC
                     ["G", 50e-6, 9006e-6], # SR400 trigger, the width is not the gate width, it is just a trigger, it can also be scanned. To realize, scan it in the loop
                    ]

# pulse_arrangement = [["B",2.5e-6,0,"TTL"],]
#                      # ["F",1,0,10.2]]

# photon counter gate width
gate_width =100e-6 

# period of a sequence
T = 1e-3

# number of the sequence cycle
cycle_number = 10

# frequency per motor step
freq_step = 78.5 #KHz

# motor step range
motor_step_range = np.arange(0,101)
relative_freq_range = freq_step * motor_step_range

# electrode setting
board_num = 0
channels = 10
bipolar = True
voltage_range = np.linspace(-5,5,101)

In [ ]:
sr400.counter_set(count_mode = "INDEPENDENT", count_preset = 0.9e-3, count_period_num=cycle_number, dwell=0, sourceA="INPUT1",sourceB="INPUT2", gate_A_mode="FIXED", gate_A_delay=0,
                gate_A_width=gate_width,gate_B_mode="CW")

In [ ]:
bnc575.disable_all()

In [ ]:
bnc575.disarm_all()
bnc575.clock_set(period=T, mode="BURST", burstctr=cycle_number)
# bnc575.clock_set(period=T,mode="CONTINUOUS")

# bnc575.channel_set(channel="H",width=pulse_arrangement[0][1],delay=pulse_arrangement[0][2], MUX="00001111")

for idn, pulse in enumerate(pulse_arrangement):
    bnc575.channel_set(channel=pulse[0],width=pulse[1],delay=pulse[2],AMP=pulse[3])

In [ ]:
bnc575.rearm_all()
bnc575.start_pulses()

In [ ]:
bg = np.mean(sr400.read_entire_counts(counter="A",length = cycle_number))
print(bg)

In [ ]:
sig = np.mean(sr400.read_entire_counts(counter="A",length = cycle_number))
print(sig)
print(bg)

In [ ]:
bnc575.disarm_all()

In [ ]:
sr400.count_reset()

# First time sequence

In [ ]:
# set the pulse sequence
#                   channel width  delay  output
# pulse_arrangement = [["A", 9000e-6, 0, "TTL"], # MOT
#                      ["C", 100e-6, 9001e-6, "TTL"], # 480 EXC
#                      ["D", 75e-6, 9106e-6, "TTL"], # DEI switch
#                      ["B", 70e-6, 9111e-6, "TTL"], # SR400 trigger, the width is not the gate width, it is just a trigger, it can also be scanned. To realize, scan it in the loop
#                     ]

pulse_arrangement = [["C", 2, 0, "TTL"], # 480 EXC
                     ["D", 2, 5e-6, "TTL"], # DEI switch
                     ["B", 70e-6, 5e-6, "TTL"]
                    ]

# pulse_arrangement = [["C", 9.9,0,"TTL"]]

# photon counter gate width
gate_width =900e-3
gate_delay = 0

# period of a sequence
# T = 10e-3
T = 3.1

# number of the sequence cycle
cycle_number = 50

# frequency per motor step
freq_step = 78.5 #KHz

# motor step range
motor_step_range = np.arange(0,101)
relative_freq_range = freq_step * motor_step_range

# electrode setting
board_num = 0
channels = 10
bipolar = True
voltage_range = np.linspace(-5,5,101)

In [ ]:
sr400.counter_set(count_mode = "INDEPENDENT", count_preset = 3, count_period_num=cycle_number, dwell=0, sourceA="INPUT1",sourceB="INPUT2", gate_A_mode="FIXED", gate_A_delay=gate_delay,
                gate_A_width=gate_width,gate_B_mode="CW")

In [ ]:
bnc575.disable_all()

In [ ]:
bnc575.disarm_all()
# bnc575.clock_set(period=T, mode="BURST", burstctr=cycle_number+1)
bnc575.clock_set(period=T,mode="CONTINUOUS")

# bnc575.channel_set(channel="H",width=pulse_arrangement[0][1],delay=pulse_arrangement[0][2], MUX="00001111")

for idn, pulse in enumerate(pulse_arrangement):
    bnc575.channel_set(channel=pulse[0],width=pulse[1],delay=pulse[2],AMP=pulse[3])

In [6]:
bnc575.rearm_all()
bnc575.start_pulses()

In [ ]:
sig = np.mean(sr400.read_entire_counts(counter="A",length = cycle_number))
print(sig)

In [ ]:
sr400.count_reset()

# verify the reading is linear

## sig_collection

In [ ]:
gate_width_list = np.linspace(1e-3,200e-3,8)
cycle_number = 10
sig_list = []
bnc575.disarm_all()
for gate_width in gate_width_list:
    sr400.counter_set(count_mode = "INDEPENDENT", count_preset = 3, count_period_num=cycle_number, dwell=0, sourceA="INPUT1",sourceB="INPUT2", gate_A_mode="FIXED", gate_A_delay=gate_delay,
                gate_A_width=gate_width,gate_B_mode="CW")
    bnc575.rearm_all()
    bnc575.start_pulses()
    time.sleep(30)
    sig = np.mean(sr400.read_entire_counts(counter="A",length = cycle_number))
    bnc575.disarm_all()
    print(sig)
    sig_list.append(sig)

## bg_collection

In [ ]:
bg_list = []
bnc575.disarm_all()
for gate_width in gate_width_list:
    sr400.counter_set(count_mode = "INDEPENDENT", count_preset = 3, count_period_num=cycle_number, dwell=0, sourceA="INPUT1",sourceB="INPUT2", gate_A_mode="FIXED", gate_A_delay=gate_delay,
                gate_A_width=gate_width,gate_B_mode="CW")
    bnc575.rearm_all()
    bnc575.start_pulses()
    time.sleep(30)
    bg = np.mean(sr400.read_entire_counts(counter="A",length = cycle_number))
    bnc575.disarm_all()
    print(bg)
    bg_list.append(bg)

In [ ]:
plt.plot(gate_width_list*1e+03,bg_list)
plt.xlabel("gate_width (ms)")
plt.ylabel("counts (BG)")

In [ ]:
plt.plot(gate_width_list*1e+03,sig_list)
plt.xlabel("gate_width (ms)")
plt.ylabel("counts (SIG)")

In [ ]:
plt.scatter(gate_width_list*1e+03,np.array(sig_list)-np.array(bg_list))
plt.xlabel("gate_width (ms)")
plt.ylabel("counts (SIG-BG)")

# Correction

In [ ]:
# set the pulse sequence
#                   channel width  delay  output
pulse_arrangement = [["C", (1e-3), 0, "TTL"], # 480 EXC
                     ["D", (1e-3), (1e-3)+5e-6, "TTL"], # DEI switch
                     ["B", 1e-6, (1e-3)+5e-6, "TTL"]
                    ]

# pulse_arrangement = [["C", 9.9,0,"TTL"]]

# photon counter gate width
gate_width =1e-6
gate_delay = 0

# period of a sequence
# T = 10e-3
T = (3e-3)

# number of the sequence cycle
cycle_number = 50

# frequency per motor step
freq_step = 78.5 #KHz

# motor step range
motor_step_range = np.arange(0,101)
relative_freq_range = freq_step * motor_step_range

# electrode setting
board_num = 0
channels = 10
bipolar = True
voltage_range = np.linspace(-5,5,101)

In [ ]:
bnc575.disable_all()

In [ ]:
bnc575.disarm_all()
# bnc575.clock_set(period=T, mode="BURST", burstctr=cycle_number+1)
bnc575.clock_set(period=T,mode="CONTINUOUS")

# bnc575.channel_set(channel="H",width=pulse_arrangement[0][1],delay=pulse_arrangement[0][2], MUX="00001111")

for idn, pulse in enumerate(pulse_arrangement):
    bnc575.channel_set(channel=pulse[0],width=pulse[1],delay=pulse[2],AMP=pulse[3])

In [ ]:
gate_width_list = np.linspace(1e-3,200e-3,8)
cycle_number = 15
sig_list = []
bnc575.disarm_all()
for gate_width in gate_width_list:
    sr400.counter_set(count_mode = "INDEPENDENT", count_preset = T-1e-6, count_period_num=cycle_number, dwell=0, sourceA="INPUT1",sourceB="INPUT2", gate_A_mode="FIXED", gate_A_delay=gate_delay,
                gate_A_width=gate_width,gate_B_mode="CW")
    bnc575.rearm_all()
    bnc575.start_pulses()
    time.sleep(20)
    sig = np.mean(sr400.read_entire_counts(counter="A",length = cycle_number))
    bnc575.disarm_all()
    print(sig)
    sig_list.append(sig)

In [ ]:
plt.plot(gate_width_list*1e+03,sig_list,".-")
plt.xlabel("gate_width (ms)")
plt.ylabel("counts (SIG)")

# Motor

In [5]:
weeder = Weeder.Weeder("ASRL1::INSTR")

In [8]:
weeder.position(header = "A", position = 0, query = False)

In [27]:
weeder.position(header = "A", query = True)

'A6300'

In [28]:
weeder.position(header = "A", query = True)

'A6300'

In [30]:
weeder.move(header = "A", position = 5000)

'A6300'

In [35]:
weeder.position(header = "A", query = True)

'A5000'

In [34]:
weeder.save(header = "A", query = True, more = True)

'saving complete.'